In [1]:
import sys
import os
 
# Obtener la ruta absoluta del directorio padre
sys.path.append(os.path.join('..'))
import torch
import pandas as pd
import numpy as np
import mne
import matplotlib.pyplot as plt
from seaborn import heatmap
from Utils.train import  patient_stratify_split, cv_patients

from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from braindecode import EEGClassifier
from braindecode.models import CBraMod, Labram
from braindecode.util import set_random_seeds
from sklearn.metrics import confusion_matrix, accuracy_score,  recall_score, precision_score, f1_score, roc_curve, auc, classification_report

# Set random seed to ensure reproducible initialization below
seed = 2024
cuda = torch.cuda.is_available()
set_random_seeds(seed=seed, cuda=cuda)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

class MyLabram(torch.nn.Module):

    def __init__(self, n_times, n_chans, n_outputs, sfreq, ch_names,
        patch_size=256, embed_dim = 128, num_layers=8, num_heads=6, mlp_ratio=4, drop_prob = 0.1,
        attn_drop_prob = 0.1, activation =torch.nn.ReLU):
        super().__init__()

        self.ch_names = ch_names

        self.model = Labram( n_times=n_times, n_chans=n_chans, n_outputs=n_outputs, num_layers=num_layers, sfreq=sfreq,
            num_heads=num_heads, patch_size=patch_size, embed_dim=embed_dim, mlp_ratio=mlp_ratio, drop_prob=drop_prob,
            attn_drop_prob=attn_drop_prob, activation=activation)

    def forward(self, x):
        return self.model(x, ch_names=self.ch_names)

# model = MyLabram(
#     n_times=n_times,
#     n_chans=n_chans,
#     n_outputs=len(outputs),
#     ch_names=channels,
#     sfreq=128,
#     patch_size=200
# )

In [2]:
original_channels=['Fp1', 'Fp2','F3','F4','C3','C4','P3','P4','O1','O2','F7','F8','T3','T4','T5','T6','Fz','Cz','Pz']
channels_dif=['Fp1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'Fp2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'F7-T3', 'T3-T5', 'T5-O1', 'F8-T4', 'T4-T6', 'T6-O2', 'Fz-Cz', 'Cz-Pz']
path_demographic="../d/data_patients.csv"
hours=[34, 36, 40 ]

############ Put the channel list
channels=original_channels
############

### Data base demographic
df_demog=pd.read_csv(path_demographic, sep="," )
df_demog["numeric_outcome"]=df_demog["Outcome"].map({"Good": 1, "Poor": 0})
df_demog.reset_index(drop=True, inplace=True)

### Ruta de donde provienen los archivos .fif, tener en cuenta que esta ruta llama las carpetas por hora.
path_files="../raw_tif/"

hour_list = os.listdir(path_files)
hour_list = [x for x in hour_list if int(x.split("h")[1]) in hours]
Raws=[]
Patients=[]
# Carga Pre-Launch de MNE asegurándose que la RAM los acople sin retrasos lazy (preload=True)
for h in hour_list:
    print(h)
    file_list = os.listdir(path_files+h+"/")
    Raws += [mne.io.read_raw_fif(path_files+h+"/"+file, preload=True, verbose=0) for file in file_list]
    Patients += [file.split(".")[0][-4:] for file in file_list]

X=[]
Scaler=StandardScaler()
demograp_data=pd.DataFrame(columns=df_demog.columns)
for k, raw in enumerate(Raws):
    patient=Patients[k]
    demograp_data=pd.concat([demograp_data,df_demog[df_demog["Id_Patient"]==int(patient)]])
    X.append(Scaler.fit_transform(raw.get_data(picks=channels)))
del Raws

t180_h34
t180_h36
t180_h40


In [3]:
X=np.c_[X][:,:, :12000]
# X=np.c_[X]
Y=np.int_(demograp_data["numeric_outcome"].values)
patients_train, patients_val= patient_stratify_split(demograp_data, train_size=0.8)
folds, Indx=cv_patients(patients_train, demograp_data, n_split=3)
n_times=X.shape[-1]
n_chans=X.shape[1]
outputs=np.unique(Y)
S_labels=["Poor", "Good"]
X= torch.as_tensor(X, dtype=torch.float32)
Y= torch.as_tensor(Y, dtype=torch.long)
resultados=[]
# del X, Y

In [ ]:
for fold, (train_idx, test_idx) in enumerate(Indx):

    print(f"\n========== FOLD {fold + 1} ==========")

    # --------------------------------
    # 1. Separar datos
    # --------------------------------
    train_dataset = TensorDataset(
    X[train_idx],
    Y[train_idx])

    val_dataset = TensorDataset(
        X[test_idx],
        Y[test_idx])

    train_loader = DataLoader(
        train_dataset,
        batch_size=16,
        shuffle=True)

    val_loader = DataLoader(
        val_dataset,
        batch_size=16,
        shuffle=False)

    # --------------------------------
    # 2. Crear NUEVO modelo
    # --------------------------------
    # model = CBraMod(n_times=n_times, n_chans=n_chans,
    #                n_outputs=len(outputs), activation=torch.nn.ReLU, sfreq=128, emb_dim=256)
    model = MyLabram( n_times=n_times, n_chans=n_chans, n_outputs=len(outputs),
                    ch_names=channels, sfreq=128, patch_size=200)
    model = model.to(device)
    
    # --------------------------------
    # 3. Crear optimizador NUEVO
    # --------------------------------
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=6e-4)
        #lr=8e-6, weight_decay=6e-7)

    criterion = torch.nn.CrossEntropyLoss()

    # --------------------------------
    # 4. Entrenamiento
    # --------------------------------

    n_epochs = 10

    for epoch in range(n_epochs):

        model.train()

        total_loss = 0
        total_batch=0
        accuraccy_fold=0
        total_true=[]
        total_score=[]
        total_pred=[]
        cont=0
        for X_batch, Y_batch in train_loader:
    
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)
    
            optimizer.zero_grad()
            output = model(X_batch)
            pred = torch.argmax(output, dim=1)
            total_true.extend(Y_batch.detach().cpu().numpy())
            total_score.extend(output[:,1].detach().cpu().numpy())
            total_pred.extend(pred.detach().cpu().numpy())

            accuraccy_fold += (pred == Y_batch).sum().item()
            total_batch += Y_batch.size(0)
            cont+=1
            loss = criterion(output, Y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        recall=recall_score(total_true, total_pred)
        f1= f1_score(total_true, total_pred)
        fpr, tpr, thresholds = roc_curve(total_true,total_score)
        auc_score=auc(fpr, tpr)
        accuraccy_fold=accuraccy_fold/total_batch
        mean_loss = total_loss / len(train_loader)
        
        
        
        model.eval()
    
        correct = 0
        total_batch = 0
        total_true=[]
        total_score=[]
        total_pred=[]
        with torch.no_grad():
            
            for X_batch, Y_batch in val_loader:
    
                X_batch = X_batch.to(device)
                Y_batch = Y_batch.to(device)
                output = model(X_batch)
                pred = torch.argmax(output, dim=1)
                total_true.extend(Y_batch.detach().cpu().numpy())
                total_score.extend(output[:,1].detach().cpu().numpy())
                total_pred.extend(pred.detach().cpu().numpy())
                cont+=1
                correct += (pred == Y_batch).sum().item()
                total_batch += Y_batch.size(0)
        val_recall=recall_score(total_true, total_pred)
        val_f1= f1_score(total_true, total_pred)
        val_accuracy = correct / total_batch
        fpr, tpr, thresholds = roc_curve(total_true,total_score)
        val_auc_score=auc(fpr, tpr)
        
        print(
            f"Fold {fold+1} | "
            f"Epoch {epoch+1}/{n_epochs} | "
            f"Loss: {mean_loss:.4f}| "
            f"Accuraccy: {accuraccy_fold:.4f}| "
            f"AUC: {auc_score:.4f}| "
            f"Recall: {recall:.4f}| "
            f"f1_score: {f1:.4f}| "
            f"Val Accuraccy: {val_accuracy:.4f}| "
            f"Val AUC: {val_auc_score:.4f}| "
            f"Val recall: {val_recall:.4f}| "
            f"Val f1: {val_f1:.4f}| ")

    # --------------------------------
    # 5. Validación
    # --------------------------------

    model.eval()

    correct = 0
    total_batch = 0
    total_true=[]
    total_score=[]
    total_pred=[]
    with torch.no_grad():
        
        for X_batch, Y_batch in val_loader:

            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)
            output = model(X_batch)
            pred = torch.argmax(output, dim=1)
            total_true.extend(Y_batch.detach().cpu().numpy())
            total_score.extend(output[:,1].detach().cpu().numpy())
            total_pred.extend(pred.detach().cpu().numpy())
            cont+=1
            correct += (pred == Y_batch).sum().item()
            total_batch += Y_batch.size(0)
            
    recall=recall_score(total_true, total_pred)
    f1= f1_score(total_true, total_pred)
    accuracy = correct / total_batch
    fpr, tpr, thresholds = roc_curve(total_true,total_score)
    auc_score=auc(fpr, tpr)
    resultados.append( (accuracy, auc_score, recall, f1))


========== FOLD 1 ==========
Fold 1 | Epoch 1/10 | Loss: 0.6855| Accuraccy: 0.5513| AUC: 0.5338| Recall: 0.1952| f1_score: 0.2714| Val Accuraccy: 0.6364| Val AUC: 0.6969| Val recall: 0.0000| Val f1: 0.0000| 
Fold 1 | Epoch 2/10 | Loss: 0.6684| Accuraccy: 0.5982| AUC: 0.5836| Recall: 0.1301| f1_score: 0.2171| Val Accuraccy: 0.5777| Val AUC: 0.6977| Val recall: 0.7984| Val f1: 0.5789| 
Fold 1 | Epoch 3/10 | Loss: 0.6163| Accuraccy: 0.6481| AUC: 0.7082| Recall: 0.5856| f1_score: 0.5876| Val Accuraccy: 0.6628| Val AUC: 0.7549| Val recall: 0.0726| Val f1: 0.1353| 
Fold 1 | Epoch 4/10 | Loss: 0.5728| Accuraccy: 0.6833| AUC: 0.7615| Recall: 0.6541| f1_score: 0.6388| Val Accuraccy: 0.7331| Val AUC: 0.7677| Val recall: 0.5806| Val f1: 0.6128| 
Fold 1 | Epoch 5/10 | Loss: 0.5175| Accuraccy: 0.7463| AUC: 0.8187| Recall: 0.6986| f1_score: 0.7022| Val Accuraccy: 0.7361| Val AUC: 0.8034| Val recall: 0.7661| Val f1: 0.6786| 


In [8]:
output

tensor([[ 0.1115, -0.1117],
        [ 0.1109, -0.1111],
        [ 0.1108, -0.1108],
        [ 0.1110, -0.1111],
        [ 0.1111, -0.1113],
        [ 0.1097, -0.1099],
        [ 0.1121, -0.1122],
        [ 0.1121, -0.1121],
        [ 0.1122, -0.1123],
        [ 0.1114, -0.1115],
        [ 0.1121, -0.1122],
        [ 0.1113, -0.1114],
        [ 0.1110, -0.1111],
        [ 0.1103, -0.1104],
        [ 0.1118, -0.1120],
        [ 0.1109, -0.1112]], device='cuda:0', grad_fn=<AddmmBackward0>)

In [5]:
total_true

[np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(1)]

In [ ]:
resultados

[(0.6081871345029239,
  0.6094902327589309,
  0.6090225563909775,
  0.5472972972972973),
 (0.6220238095238095,
  0.600658275462963,
  0.4722222222222222,
  0.5171102661596958),
 (0.6076696165191741,
  0.6366906474820144,
  0.5899280575539568,
  0.5521885521885522)]